# Well Played, Mauer: Mathematical OPS

What would happen if we properly combined **OBP** and **SLG** mathematically?

In [1]:
# import the necessary packages
import os
import sys
import pandas as pd

In [2]:
# set up the file paths
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
raw_data_dir = os.path.join(project_root, 'data', 'raw')
processed_data_dir = os.path.join(project_root, 'data', 'processed')

# test the paths
# print(f'Project Root: {project_root}')
# print(f'Raw Data Directory: {raw_data_dir}')
# print(f'Processed Data Directory: {processed_data_dir}')

In [3]:
# read in the csv for all qualified seasons from 2006 - 2015
# data courtesy of stathead
filename = 'mlb_qualified_batters_2006_2015.csv'
csv_path = os.path.join(raw_data_dir, filename)
batters = pd.read_csv(csv_path)
batters.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1495 entries, 0 to 1494
Data columns (total 40 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Rk                 1495 non-null   int64  
 1   Player             1495 non-null   object 
 2   Season             1495 non-null   int64  
 3   Age                1495 non-null   int64  
 4   Team               1495 non-null   object 
 5   Lg                 1495 non-null   object 
 6   G                  1495 non-null   int64  
 7   PA                 1495 non-null   int64  
 8   AB                 1495 non-null   int64  
 9   R                  1495 non-null   int64  
 10  H                  1495 non-null   int64  
 11  1B                 1495 non-null   int64  
 12  2B                 1495 non-null   int64  
 13  3B                 1495 non-null   int64  
 14  HR                 1495 non-null   int64  
 15  RBI                1495 non-null   int64  
 16  SB                 1495 

The idea is to combine **OBP** and **SLG** by scaling each fraction so that it has a common denominator with the other. I will be calling this **mathematical OPS (mOPS)**. The math behind that would be as follows:

$$ \begin{align} \text{mOPS} &= \text{OBP} + \text{SLG} \\ 
&= \frac{\text{H} + \text{BB} + \text{HBP}}{\text{AB} + \text{BB} + \text{HBP} + \text{SF}} + \frac{\text{TB}}{\text{AB}} \\ 
&= \frac{\text{AB}(\text{H} + \text{BB} + \text{HBP})}{\text{AB}(\text{AB} + \text{BB} + \text{HBP} + \text{SF})} + \frac{\text{TB}(\text{AB} + \text{BB} + \text{HBP} + \text{SF})}{\text{AB}(\text{AB} + \text{BB} + \text{HBP} + \text{SF})} \end{align} $$

In [4]:
# define the numerator and denominator of OBP
batters['OBP_num'] = batters['H'] + batters['BB'] + batters['HBP']
batters['OBP_den'] = batters['AB'] + batters['BB'] + batters['HBP'] + batters['SF']

In [5]:
# calculate mathematical OPS (mOPS)
batters['mOPS'] = (batters['AB'] * batters['OBP_num'] + batters['TB'] * batters['OBP_den']) / (batters['AB'] * batters['OBP_den'])

In [6]:
# find the difference between standard and mathematical OPS
batters['OPS - mOPS'] = batters['OPS'] - batters['mOPS']

In [7]:
display_cols = ['Player', 'Season', 'BA', 'OBP', 'SLG', 'OPS', 'mOPS', 'OPS - mOPS']
batters[display_cols].sort_values(by='OPS - mOPS', ascending=False).head(10)

,Player,Season,BA,OBP,SLG,OPS,mOPS,OPS - mOPS
733,Ender Inciarte,2015,0.303,0.338,0.408,0.747,0.746501,0.000499
504,Daniel Murphy,2012,0.291,0.332,0.403,0.735,0.734501,0.000499
779,Ben Revere,2015,0.306,0.342,0.377,0.719,0.718502,0.000498
328,Shane Victorino,2011,0.279,0.355,0.491,0.847,0.846502,0.000498
1228,Michael Young,2013,0.279,0.335,0.395,0.730,0.729504,0.000496
1303,Jed Lowrie,2014,0.249,0.321,0.355,0.676,0.675504,0.000496
1326,Alcides Escobar,2014,0.285,0.317,0.377,0.694,0.693505,0.000495
981,Luis Gonzalez,2007,0.278,0.359,0.433,0.793,0.792505,0.000495
109,Carlos Quentin,2008,0.288,0.394,0.571,0.965,0.964506,0.000494
1111,Rafael Furcal,2009,0.269,0.335,0.375,0.711,0.710507,0.000493


In [8]:
batters[display_cols].sort_values(by='OPS - mOPS', ascending=True).head(10)

,Player,Season,BA,OBP,SLG,OPS,mOPS,OPS - mOPS
403,Carlos Peña,2011,0.225,0.357,0.462,0.819,0.819499,-0.000499
1072,Aramis Ramírez,2010,0.241,0.294,0.452,0.745,0.745499,-0.000499
459,Ryan Ludwick,2009,0.265,0.329,0.447,0.775,0.775498,-0.000498
1401,Ichiro Suzuki,2012,0.283,0.307,0.390,0.696,0.696498,-0.000498
77,Joe Mauer,2012,0.319,0.416,0.446,0.861,0.861497,-0.000497
1325,Casey Blake,2007,0.270,0.339,0.437,0.776,0.776496,-0.000496
655,Coco Crisp,2013,0.261,0.335,0.444,0.779,0.779496,-0.000496
970,Mike Moustakas,2012,0.242,0.296,0.412,0.708,0.708495,-0.000495
370,Carlos Gómez,2014,0.284,0.356,0.477,0.833,0.833495,-0.000495
715,Nate Schierholtz,2013,0.251,0.301,0.470,0.770,0.770494,-0.000494


It doesn't really seem to change much.